In [41]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

In [66]:
x, y = sp.symbols('x y')
f = sp.exp(x) + sp.exp(y)

display_rad = 2
display_shift = 0
x_display = (x, -display_rad + display_shift, display_rad + display_shift)
y_display = (y, -display_rad + display_shift, display_rad + display_shift)

In [67]:
dfdx = sp.diff(f, x)
dfdy = sp.diff(f, y)
print(dfdx, dfdy)

d2fdx2 = sp.diff(dfdx, x)
d2fdydx = sp.diff(dfdx, y)
print(d2fdx2, d2fdydx)

d2fdy2 = sp.diff(dfdy, y)
d2fdxdy = sp.diff(dfdy, x)
print(d2fdy2, d2fdxdy)

H = sp.Matrix([[d2fdx2, d2fdxdy], [d2fdydx, d2fdy2]])
print(H)

exp(x) exp(y)
exp(x) 0
exp(y) 0
Matrix([[exp(x), 0], [0, exp(y)]])


In [75]:
# We cannot use Sympy's diagonalize() because it does not find an orthogonal Q
# Instead, we perform manual orthogonal diagonalization

# Find the eigenvalues of the 2x2 matrix
eigs = H.eigenvals(multiple=True)
lamda_1 = eigs[0]
lamda_2 = eigs[1]
lamda = sp.diag(lamda_1, lamda_2)
print(lamda)

Matrix([[exp(x), 0], [0, exp(y)]])


In [79]:
# Find the positive and negative components of the eigenvalues
lamda_1_plus = sp.simplify((lamda_1 + sp.Abs(lamda_1)) / 2)
lamda_2_plus = sp.simplify((lamda_2 + sp.Abs(lamda_2)) / 2)

lamda_1_minus = sp.simplify(((- lamda_1) + sp.Abs(- lamda_1)) / 2)
lamda_2_minus = sp.simplify(((- lamda_2) + sp.Abs(- lamda_2)) / 2)

lamda_minus = sp.diag(lamda_1_minus, lamda_2_minus)
lamda_plus = sp.diag(lamda_1_plus, lamda_2_plus)
print(lamda_minus)
print(lamda_plus)

Matrix([[-exp(x)/2 + exp(re(x))/2, 0], [0, -exp(y)/2 + exp(re(y))/2]])
Matrix([[exp(x)/2 + exp(re(x))/2, 0], [0, exp(y)/2 + exp(re(y))/2]])


In [69]:
# Use the characteristic polynomials (H-lambda_1*I)x_1 = 0 (H-lambda_2*I)x_2 = 0
# Find the eigenvectors
A = H - lamda_1 * sp.eye(H.shape[0])
B = H - lamda_2 * sp.eye(H.shape[0])
A_null = A.nullspace()
B_null = B.nullspace()

x_1 = A_null[0]
x_2 = B_null[0]
print(x_1)
print(x_2)
print(x_1.shape)
print(x_2.shape)


Matrix([[1], [0]])
Matrix([[0], [1]])
(2, 1)
(2, 1)


In [70]:
# Use the Gram-Schmidt process to orthonormalize the vectors
# Write my own custom process for 2x2 to keep things simple

# u is a 2x1 vector
def normalize(u):
    numerator = u
    denominator = sp.sqrt(u[0] ** 2 + u[1] ** 2)
    return numerator / denominator

# u, v are 2x1 vectors
def proj(u, v):
    numerator = v.dot(u)
    denominator = u.dot(u)
    return (numerator / denominator) * u

u_1 = x_1
e_1 = normalize(u_1)

u_2 = (x_2 - proj(u_1, x_2))
e_2 = normalize(u_2)

print(e_1)
print(e_2)
print(e_1.dot(e_2))

Matrix([[1], [0]])
Matrix([[0], [1]])
0


In [71]:
# Put these orthonormal vectors into the columns of a matrix
# Need to do some hoop-jumping as Sympy matrices are defined by row

Q_T = sp.Matrix([e_1.T, e_2.T])
Q = Q_T.T
print(Q.shape)

(2, 2)


In [74]:
print(H)
print(Q * lamda * Q.T)

Matrix([[exp(x), 0], [0, exp(y)]])
Matrix([[exp(x), 0], [0, exp(y)]])


In [83]:
H_minus = Q * lamda_minus * Q.T 
H_plus = Q * lamda_plus * Q.T

print(H)
print(H_plus - H_minus)

Matrix([[exp(x), 0], [0, exp(y)]])
Matrix([[exp(x), 0], [0, exp(y)]])
